In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import json
from typing import List, Dict

CUAD_PATH = r"/content/drive/MyDrive/Colab Notebooks/Fine Tuning/CUAD_v1.json"
MODEL_NAME = "meta-llama/Llama-2-7b-hf"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Fine Tuning/raft_qlora_outputv212"

# QLoRA Configuration
QLORA_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": torch.float16,
    "bnb_4bit_use_double_quant": True,
}

LORA_CONFIG = {
    "r": 64,                              # LoRA rank
    "lora_alpha": 16,                     # LoRA alpha (scaling)
    "lora_dropout": 0.05,                 # Dropout for LoRA layers
    "bias": "none",                       # No bias
    "task_type": "CAUSAL_LM",            # Task type
    "target_modules": [                   # Modules to apply LoRA to
        "q_proj",
        "v_proj",
        "k_proj",
        "o_proj",
        "up_proj",
        "down_proj",
        "gate_proj"
    ],
}

def get_bnb_config():

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=QLORA_CONFIG["load_in_4bit"],
        bnb_4bit_quant_type=QLORA_CONFIG["bnb_4bit_quant_type"],      # NF4 = normal float 4
        bnb_4bit_compute_dtype=QLORA_CONFIG["bnb_4bit_compute_dtype"],# Compute in FP16
        bnb_4bit_use_double_quant=QLORA_CONFIG["bnb_4bit_use_double_quant"],  # Double quantization
    )
    return bnb_config

def get_lora_config():

    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],                                    # Rank of LoRA
        lora_alpha=LORA_CONFIG["lora_alpha"],                 # Scaling factor
        target_modules=LORA_CONFIG["target_modules"],         # Which modules to adapt
        lora_dropout=LORA_CONFIG["lora_dropout"],            # Regularization
        bias=LORA_CONFIG["bias"],                             # No bias adaptation
        task_type=LORA_CONFIG["task_type"],                   # Task type
    )
    return lora_config

def load_model_with_qlora(model_name: str = MODEL_NAME):

    print("[1/3] Loading 4-bit quantization config...")
    bnb_config = get_bnb_config()
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )

    print(f"✓ Model loaded successfully")
    print(f"  - Model size: {model.get_memory_footprint() / 1e9:.1f} GB (quantized)")

    # Prepare model for kbit training
    model = prepare_model_for_kbit_training(model)
    print(f"✓ Model prepared for kbit training")

    # Apply LoRA
    lora_config = get_lora_config()
    model = get_peft_model(model, lora_config)

    print(f"✓ LoRA adapters applied")
    print(f"  - Trainable parameters: {model.print_trainable_parameters()}")

    return model

class RAFTDataProcessorQLoRA:

    def __init__(self):
        self.train_examples = []

    def extract_passages(self, context: str, chunk_size: int = 256, overlap: int = 50) -> List[str]:
        passages = []
        words = context.split()

        for i in range(0, len(words), chunk_size - overlap):
            passage = ' '.join(words[i:i + chunk_size])
            if passage.strip():
                passages.append(passage)

        return passages
    def process_cuad(self, data):
        examples = []

        for sample in data["train"]:
            for doc in sample["data"]:
                for paragraph in doc["paragraphs"]:

                    context = paragraph["context"]

                    for qa in paragraph["qas"]:

                        question = qa["question"]

                        if len(qa["answers"]) == 0:
                            continue

                        answers = [a["text"] for a in qa["answers"]]

                        # Extract passages
                        passages = self.extract_passages(context)

                        # Identify relevant passages
                        relevant_passages = []
                        irrelevant_passages = []

                        for passage in passages:
                            is_relevant = any(
                                ans.lower() in passage.lower()
                                for ans in answers
                            )

                            if is_relevant:
                                relevant_passages.append(passage)
                            else:
                                irrelevant_passages.append(passage)

                        # Create training example for each answer
                        for answer in answers:

                            retrieved = (
                                relevant_passages[:3] +
                                irrelevant_passages[:2]
                            )

                            example = {
                                "question": question,
                                "context": " ".join(retrieved),
                                "answer": answer,
                            }

                            examples.append(example)

        self.train_examples = examples
        return examples
def format_for_qlora(examples: List[Dict], tokenizer) -> List[Dict]:
    formatted = []

    for example in examples:
        # Create prompt
        prompt = f"""[INST] Based on the following contract information, answer the question.

Contract Information:
{example['context'][:500]}

Question: {example['question']} [/INST]"""

        # Create full text
        full_text = prompt + f" {example['answer']}"

        # Tokenize
        encoding = tokenizer(
            full_text,
            truncation=True,
            max_length=512,
            padding='max_length',
            return_tensors='pt'
        )

        formatted.append({
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': encoding['input_ids'].squeeze(),
        })

    return formatted

def train_raft_qlora():

    print("RAFT FINE-TUNING WITH QLORA + PEFT")
    print("[STEP 1/6] Loading CUAD dataset...")
    data = load_dataset("json", data_files=CUAD_PATH)
    print(data)
    print(data["train"][0])
    print(f"✓ Loaded {len(data['train'])} samples")

    print("\n[STEP 2/6] Processing data for RAFT...")
    processor = RAFTDataProcessorQLoRA()
    train_examples = processor.process_cuad(data)
    print(f"✓ Created {len(train_examples)} training examples")

    print("\n[STEP 3/6] Loading model with QLoRA...")
    model = load_model_with_qlora(MODEL_NAME)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    print("✓ Tokenizer loaded")

    print("\n[STEP 4/6] Formatting and tokenizing data...")
    formatted_examples = format_for_qlora(train_examples, tokenizer)
    train_dataset = Dataset.from_list(formatted_examples)
    print(f"✓ Dataset ready with {len(train_dataset)} examples")

    print("\n[STEP 5/6] Setting up training arguments...")
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=1,              # Can be larger with QLoRA
        gradient_accumulation_steps=2,
        gradient_checkpointing=True,          # Effective batch = 16
        save_steps=500,
        save_total_limit=3,
        logging_steps=100,
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_steps=100,
        fp16=True,                                  # Mixed precision
        optim="paged_adamw_8bit",                   # 8-bit optimizer (saves memory)
        max_grad_norm=1.0,
        report_to=["tensorboard"],
    )
    print("✓ Training arguments configured")
    print("\n[STEP 6/6] Starting RAFT QLoRA fine-tuning...")
    print("Training...\n")

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    trainer.train(
        resume_from_checkpoint="/content/drive/MyDrive/Colab Notebooks/Fine Tuning/raft_qlora_outputv212/checkpoint-6000"
    )
    print("✅ TRAINING COMPLETED SUCCESSFULLY!")
    print("\nSaving model...")
    model.save_pretrained(f"{OUTPUT_DIR}/final_qlora_model")
    tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_qlora_model")
    print(f"✓ LoRA model saved to: {OUTPUT_DIR}/final_qlora_model")

    return model, tokenizer

def inference_with_qlora(model_path: str, tokenizer_path: str):

    from peft import AutoPeftModelForCausalLM
    print("INFERENCE WITH QLORA MODEL")

    # Load model with LoRA adapters
    print("\nLoading QLoRA model...")
    model = AutoPeftModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    tokenizer.pad_token = tokenizer.eos_token
    print("✓ Model loaded")

    # Test inference
    test_question = "What is the contract duration?"
    test_context = """This agreement is effective from January 1, 2024, and will continue
    for a period of two years unless terminated earlier."""

    prompt = f"""[INST] Based on the following contract information, answer the question.

Contract Information:
{test_context}

Question: {test_question} [/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=150,
            temperature=0.7,
            do_sample=True,
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = answer.replace(prompt, "").strip()

    print(f"\nQuestion: {test_question}")
    print(f"Generated Answer: {answer}")

    return model

train_raft_qlora()


In [ ]:
import os
os.listdir("/content/drive/MyDrive/Fine Tuning/raft_qlora_outputv212/checkpoint-6912")

In [ ]:
import json

with open("/content/drive/MyDrive/Fine Tuning/raft_qlora_outputv212/checkpoint-6912/trainer_state.json", "r") as f:
    trainer_state = json.load(f)
print(trainer_state.keys())
log_history = trainer_state["log_history"]

print(log_history[:5])
steps = []
losses = []

for log in log_history:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])
import matplotlib.pyplot as plt
plt.plot(steps,losses,linewidth=2)
plt.xlabel("Training Steps")
plt.ylabel("Training Losses")
plt.grid(True)
plt.show()

In [ ]:
!pip install -U langchain-community langchain-huggingface langchain-text-splitters sentence-transformers faiss-cpu

In [ ]:
import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
from pdf2image import convert_from_path
import pytesseract
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time

test_context = []
times = []
pdf = "/content/drive/MyDrive/Fine Tuning/SEC contract.pdf"
def OCR1(pdf):
  pages = convert_from_path(pdf,dpi=300)
  for i,pagei in enumerate(pages):
    ft = pytesseract.image_to_string(pagei)
    test_context.append(f"--- PAGE {i + 1} ---\n{ft}\n")
  return'\n'.join(test_context)

test_context = OCR1(pdf)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=250,
    length_function=len
)
chunks = text_splitter.split_text(test_context)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_texts(chunks, embeddings)
retriever = db.as_retriever(
    search_kwargs={"k": 5}
)

# Model path
MODEL_PATH = "/content/drive/MyDrive/Fine Tuning/raft_qlora_outputv212/final_qlora_model"

print("Loading QLoRA model...")

# Load model
model = AutoPeftModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

print("✓ Model Loaded Successfully")

questions = [
"Who are the parties to the agreement?",
"What is the effective date of the agreement?",
"What is the purpose of the agreement?",
"What is the contract duration?",
"What are the key responsibilities?",
"What is the payment or consideration?",
"What is the annual base salary or compensation?",
"Can the agreement be renewed?",
"Under what conditions can the agreement be terminated?",
"Does the agreement include a confidentiality clause?",
"Is there a non-compete clause?",
"What is the governing law?",
"How will disputes be resolved?",
"Who owns the intellectual property?",
"What are the key obligations of each party?"]

for test_question in questions:
  start = time.time()
  match1 = retriever.invoke(test_question)
  ret_context = "\n\n".join(
    [doc.page_content for doc in match1]
)
  prompt = f"""[INST]
  Based on the following contract information, answer the question.

  Answer ONLY using information explicitly present in the contract.

  Contract Information:
  {ret_context}

  Question:
  {test_question}
  [/INST]"""

  # Tokenize
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  # Generate Answer
  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=30,
          do_sample=False,
          repetition_penalty=1.1,
          pad_token_id=tokenizer.eos_token_id,
      )

  # Decode
  generated_text = tokenizer.decode(
      outputs[0][inputs["input_ids"].shape[1]:],
      skip_special_tokens=True
  ).strip()
  end = time.time()
  times.append(end-start)
for q, t in zip(questions, times):
  print(f"{q} : {t:.3f} seconds")
import matplotlib.pyplot as plt

# Question numbers
question_numbers = [f"Q{i}" for i in range(1, len(times) + 1)]

plt.figure(figsize=(10,6))

plt.bar(question_numbers, times)

plt.title("Inference Time per Question")
plt.xlabel("Question Number")
plt.ylabel("Inference Time (seconds)")

plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()

plt.savefig("Inference_Time_Per_Question.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
!pip install torchao

In [ ]:
!pip uninstall -y pillow
!pip install pillow==10.4.0

In [ ]:
!apt-get update -qq
!apt-get install -y poppler-utils

In [ ]:
!pip install pytesseract pdf2image Pillow


In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

In [ ]:
!pip install -U transformers==4.52.4 peft==0.19.1 accelerate bitsandbytes

In [ ]:
import transformers
import peft
import torch

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)

In [ ]:
!pip install -U torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')